
<div class="alert alert-danger", style=" padding: 20px;font-size:20px;">
<center> <b>1.0 Ingest
</b></center>     
</div>


<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b>Load Data （当前只支持Pdf）
</b></center> 

</div>


In [1]:

from tiny_rag.ingest.loaders import load



In [2]:
pages = load("test_data/test.pdf")


In [3]:
print(pages[0].keys())


dict_keys(['text', 'page', 'source', 'filename', 'title'])


In [4]:
print(pages[0]['filename'])
print(pages[0]['title'])
print(pages[0]['source'])
print(pages[0]['page'])

test.pdf
浙江万里学院硕士学位授予单位
/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf
1


In [5]:
print(pages[0]['text'][0:50])

浙万院人“2022”13 号 

## 关于印发浙江万里学院第五轮岗位设置 与聘任工作指导意见的通知



<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b>Chunk
</b></center> 

</div>


In [6]:
from tiny_rag.ingest.chunker import chunk_pages
import re

"""Split a list of page dicts into chunks.

    Args:
        pages: output of loaders.load_pdf — list of
               {"text", "page", "source", "filename", "title"}
        chunk_size: max tokens per chunk (default: settings.chunk_size, in bge-m3 tokens)
        chunk_overlap: overlap tokens between consecutive chunks (default: settings.chunk_overlap)

    Returns:
        list of chunk dicts, each:
            {
                "text": str,
                "page": int,
                "source": str,
                "filename": str,
                "title": str,
                "chunk_index": int,        # within this page
                "chunk_id": str,           # unique: source#p{page}#c{index}
                "token_count": int,        # bge-m3 tokens
            }
"""

objc[6304]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x32d5543a8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3344c43a8). One of the two will be used. Which one is undefined.
objc[6304]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x32d5543f8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3344c43f8). One of the two will be used. Which one is undefined.


'Split a list of page dicts into chunks.\n\n    Args:\n        pages: output of loaders.load_pdf — list of\n               {"text", "page", "source", "filename", "title"}\n        chunk_size: max tokens per chunk (default: settings.chunk_size, in bge-m3 tokens)\n        chunk_overlap: overlap tokens between consecutive chunks (default: settings.chunk_overlap)\n\n    Returns:\n        list of chunk dicts, each:\n            {\n                "text": str,\n                "page": int,\n                "source": str,\n                "filename": str,\n                "title": str,\n                "chunk_index": int,        # within this page\n                "chunk_id": str,           # unique: source#p{page}#c{index}\n                "token_count": int,        # bge-m3 tokens\n            }\n'

In [7]:
chunks=chunk_pages(pages)

In [8]:
len(chunks)

118

In [9]:
chunks[0].keys()

dict_keys(['source', 'filename', 'title', 'page', 'text', 'chunk_index', 'chunk_id', 'token_count'])

In [10]:
print(chunks[0]['source'])
print(chunks[0]['title'])
print(chunks[0]['chunk_index'])
print(chunks[3]['chunk_id'])


/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf
浙江万里学院硕士学位授予单位
0
/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf#p4#c0


In [11]:
print(chunks[0]['text'])


浙万院人“2022”13 号 

## 关于印发浙江万里学院第五轮岗位设置 与聘任工作指导意见的通知 

各单位、部门： 

现将《浙江万里学院第五轮岗位设臵与聘任工作指导意见》 印发给你们，请结合实际，认真贯彻执行。 

浙江万里学院 2022 年5 月30 日 

-1-



<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b>embedder
</b></center> 

</div>


In [12]:
from tiny_rag.ingest.embedder import embed_chunks, embed_query


In [13]:
vectors = embed_chunks(chunks)     # 100×1024 矩阵

Loading embedding model: BAAI/bge-m3
  (first run downloads ~2GB, subsequent runs use HF cache)
  Device: cpu


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [14]:
vectors.shape

(118, 1024)

In [20]:
q_vec = embed_query("两种心智能力是什么？")   # shape (1024,)


In [21]:
q_vec.shape

(1024,)


<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b>store 存储
</b></center> 

</div>


In [17]:
from tiny_rag.ingest.store import add_chunks, query,delete_by_source,list_sources,count


In [18]:
add_chunks(chunks,vectors)

118

In [22]:
list_sources()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


['/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf',
 '/Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_structured.pdf']

In [19]:
count()

120

In [24]:
hits=query(q_vec,50)

In [29]:
hits[0:1]

[{'id': '/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf#p60#c0',
  'text': '<!-- Start of picture text -->\n姓 名 性 别 出生年月<br>现所属部门 参加工作时间 专业技术职务<br>现聘任 现聘任 现聘任<br>岗位名称 岗位等级 岗位时间<br>上一轮聘期考核情况 近四年学年考核情况<br>应聘岗位名称 应聘岗位等级<br>一、应聘岗位职责及任期目标（注：岗位职责及任期目标由部门制定。）<br><!-- End of picture text -->\n\n-60-',
  'metadata': {'chunk_index': 0,
   'filename': 'test.pdf',
   'page': 60,
   'source': '/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf',
   'title': '浙江万里学院硕士学位授予单位',
   'token_count': 127},
  'distance': 0.35941428657068497}]


<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b>sync
</b></center> 

</div>


## sync 实现的 4 个核心

**1. state.json 存所有的文件状态**（指纹 + 元数据）

```json
{
  "/abs/path/to/book.pdf": {
    "mtime": 1704700000.0,        // 文件修改时间
    "size": 12345,                // 文件大小
    "content_hash": "abc123...",   // SHA1 前 16 字符
    "chunk_count": 42,            // 入了多少 chunk
    "ingested_at": "2024-01-15T..." // 入库时间
  },
 "/abs/path/to/book2.pdf": {
    xxx
     xxx
  },
}
```

**2. 用 mtime + size 快速检测变化**（99% case）

```python
def is_unchanged(path, record):
    s = path.stat()
    if s.st_mtime != record["mtime"]:  # mtime 不同 = 改了
        return False
    if s.st_size != record["size"]:    # size 不同 = 改了
        return False
    return True
```

**为什么不用 content_hash**：
- mtime + size 8 字节 vs SHA1 要读完整文件（几 MB 可能）
- 编辑器保存文件**一定**改 mtime（99% case）
- 极少数「mtime 不变但内容变」的情况（编辑器 bug / 拷贝保留 mtime），等下次 mtime 真改了再 catch

**3. chunk_id 稳定 = 重新入库等于覆盖，不是插入**

之前 chunker 设计：`chunk_id = source#page#index`
- 重新 ingest 同文件 → Chroma upsert（同 id 覆盖）✓
- 不需要「先 delete 再 insert」两步

**4. 三种动作**：skip / ingest / remove

```
sync_file(path):
  ├─ 文件存在 + state 有 + mtime/size 匹配    → skip
  ├─ 文件存在 + state 有 + mtime/size 不匹配  → delete + re-ingest
  ├─ 文件存在 + state 没有                     → ingest (新的)
  └─ 文件不存在 + state 有                    → delete chunks + remove from state
```

---

## 完整 sync_file 流程

```python
def sync_file(path, force=False):
    path = path.resolve()
    state = load_state()  # 从 state.json 读
    record = state.get(str(path))

    if not path.exists():
        if record:
            # 文件被删了 → 清理
            delete_by_source(str(path))  # 删 Chroma chunks
            remove_from_state(path)       # 删 state 记录
        return {"action": "removed"}

    if record and not force and is_unchanged(path, record):
        return {"action": "skipped"}

    # 新文件或改了：先清旧 chunks 再入库
    if record:
        delete_by_source(str(path))
    n = ingest_file(path)       # load → chunk → embed → store
    update_state(path, n)      # 更新 state.json
    return {"action": "ingested"}
```

---

## sync_directory 多了「处理删除」

```python
def sync_directory(directory, pattern, recursive, force):
    files = sorted(directory.glob(pattern))
    state = load_state()
    state_keys = set(state.keys())
    dir_keys = {str(f.resolve()) for f in files}

    deleted_keys = state_keys - dir_keys  # state 有但目录里没了

    # 1. 清理已删除文件
    for key in deleted_keys:
        delete_by_source(key)
        remove_from_state(key)

    # 2. 处理目录里现有文件
    for f in files:
        sync_file(f, force=force)
```

**这一段是「incidental cleanup」**——平时 `sync_file` 一次只处理一个文件，看不到「其他文件被删了」。`sync_directory` 才有机会做整目录 diff。

---

## 关键设计决策

### 1. state 用 absolute path 当 key

```python
key = str(path.resolve())  # 永远用绝对路径
```

避免相对路径 + cwd 变化导致 state key 不一致。

### 2. chunk_count 存进 state

```python
"chunk_count": n  # ingest 后记下来
```

下次 sync 跳过时，可以直接返回「chunks=N」给用户显示，不用再去数据库数。

### 3. 模块级 state cache

```python
_state_cache: Dict = None  # 模块级单例

def load_state():
    global _state_cache
    if _state_cache is not None:
        return _state_cache  # 一次加载多次用
    # ... load from disk
```

sync 一次会 sync 多个文件，state 不会反复 IO。

### 4. fail-safe 行为

```python
# delete 先于 ingest（虽然 upsert 也能覆盖，但更稳）
if record:
    delete_by_source(str(path))  # 1. 先清旧
n = ingest_file(path)            # 2. 再入库
update_state(path, n)            # 3. 更新 state
```

如果 ingest 中途失败，state 没更新，下次 sync 会重试。**不会留下「半成品」state**。

### 5. content_hash 存了但没用于判断

```python
"content_hash": compute_hash(path)  # 算并存了
# 但 is_unchanged 只看 mtime + size
```

content_hash 主要用于：
- 调试（看 state.json 知道每个文件实际内容指纹）
- 未来想升级到 hash-only 判定时数据已就位

---

## sync 三个阶段

**Phase A — 轻量检查**（mtime + size）：
```python
if is_unchanged(path, record):  # 只读 stat，0 IO
    return "skipped"
```

**Phase B — 删旧 chunk**（Chroma delete）：
```python
delete_by_source(path)  # Chroma 删 N 个 chunks
```

**Phase C — 重新入库**（load + chunk + embed + store）：
```python
n = ingest_file(path)  # 最贵的一步
```

99% 文件走 Phase A 直接 skip（秒级）。1% 改过的走 Phase B + C（分钟级）。

---

## 跟 git 的对比

| git | Tiny_RAG sync |
|---|---|
| `.git/index` | `data/state.json` |
| 文件 hash（SHA1）| 文件 hash（SHA1）+ mtime + size |
| `git status` | `python -m tiny_rag status` |
| `git add` | `sync_file` |
| `git add -u` | `sync_directory`（自动处理删除）|

本质都是「本地文件状态管理」，但 sync 没有「commit / push / branch」这些概念，更简单。

---

## 一句话总结

**sync = state.json 记指纹 + 对比 mtime/size 决定 skip/ingest + 自动清理删除**——3 个动作（skip/ingest/remove）覆盖 99% 增量场景。

In [74]:
# 增量同步（推荐每天用）
! python -m tiny_rag sync /Users/sunshine8641/Code/Tiny_RAG/test_data/ --recursive

# 强制全量重建
# ! python -m tiny_rag sync ~/books --recursive --force

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


objc[95323]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x16f3b83a8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x301c243a8). One of the two will be used. Which one is undefined.
objc[95323]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x16f3b83f8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x301c243f8). One of the two will be used. Which one is undefined.
Cleaning 1 deleted file(s)...
Syncing:   0%|                                            | 0/2 [00:00<?, ?it/s]
→ Ingesting: test.pdf
  Loaded:  86 pages
  Chunked: 118 chunks
Loading embedding model: BAAI/bge-m3
  (first run downloads ~2GB, subsequent runs use HF cache)
  Device: cpu
  Embedded: 118 vectors of dim 1024
  ✓ Sto


<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b> 最终pipeline
</b></center> 

</div>


In [45]:
from tiny_rag.ingest.pipeline import ingest_directory, ingest_file

```python
# 假设 books/ 下有：
#   books/a.pdf
#   books/sub/b.pdf
# recursive=False (用 glob)
p.glob("*.pdf")
# → [PosixPath('books/a.pdf')]
# （只搜 books/ 这层，sub/b.pdf 没找到）
# recursive=True (用 rglob)
p.rglob("*.pdf")
# → [PosixPath('books/a.pdf'), PosixPath('books/sub/b.pdf')]
# （递归搜所有子目录）

```

In [46]:
ingest_file('/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf')


→ Ingesting: test.pdf


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Loaded:  86 pages
  Chunked: 118 chunks
  Embedded: 118 vectors of dim 1024
  ✓ Stored: 118 chunks


118

In [49]:
list_sources()

['/Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf',
 '/Users/sunshine8641/Code/Tiny_RAG/test_data/人心的本质.pdf',
 '/Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf',
 '/Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_structured.pdf']

In [48]:
ingest_directory('/Users/sunshine8641/Code/Tiny_RAG/test_data/')

Found 2 files in /Users/sunshine8641/Code/Tiny_RAG/test_data


Ingesting:   0%|                                                              | 0/2 [00:00<?, ?it/s]


→ Ingesting: test.pdf


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Loaded:  86 pages
  Chunked: 118 chunks


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Loaded:  279 pages
  Chunked: 478 chunks
  Embedded: 478 vectors of dim 1024


Ingesting: 100%|██████████████████████████████████████████████████████| 2/2 [02:45<00:00, 82.85s/it]

  ✓ Stored: 478 chunks

✓ Done: 2/2 files ingested, 596 total chunks


{'test.pdf': 118, '人心的本质.pdf': 478}

In [50]:
q_vec = embed_query("两种心智能力是什么？")   # shape (1024,)


In [51]:
hits=query(q_vec,50)

In [53]:
hits[0]

{'id': '/Users/sunshine8641/Code/Tiny_RAG/test_data/人心的本质.pdf#p23#c1',
 'text': '总之，被调查者需要⽐较13个潜在⼼智俱乐部成员之间不同⼼智 能⼒⽅⾯的特点及差异。例如，其中⼀个问题是希尔感受到的痛苦会 ⽐托⽐感受到的多还是少。顺便提⼀下，⼤多数⼈的回答是“多”。我 们计算了每种⼼智能⼒在所有特点上的平均评分，然后⽤因素分析法 判断⼼智能⼒的聚集性。⼈们对隐秘⼼智在各种⼼智能⼒上的评级都 是从“最少”到“最多”吗？ \n\n**事实证明，⼼智能⼒并不都聚集在⼀起。相反，⼈们是以两种根 本不同的因素为依据来查看⼼智的，我们将这两种⼼智能⼒称为感受 性（experience）和能动性（agency）**<sup>13</sup> **。** \n\n感受性因素体现的是存在体具有内⼼⽣活、情感和感受的能⼒。 它包括感到饥饿、恐惧、痛苦、快乐、愤怒和渴望的能⼒，还包括⼈ 格、意识、骄傲、尴尬和喜悦等⼼理表现。⼼智的这些⽅⾯似乎体现 了拥有⼼智是什么样⼦的，这是当⼼理学家、哲学家探讨意识的难题',
 'metadata': {'chunk_index': 1,
  'filename': '人心的本质.pdf',
  'page': 23,
  'source': '/Users/sunshine8641/Code/Tiny_RAG/test_data/人心的本质.pdf',
  'title': '人心的本质（中国社会心理学会2020年度心理学好书，一套理解复杂人性和行为的极简思维工具，彻底改变你看待世界和自己的方式，“谷歌效应”提出者、社会心理学泰斗丹尼尔·韦格纳绝笔之作。）',
  'token_count': 262},
 'distance': 0.39879143238067627}


<div class="alert alert-danger", style=" padding: 20px;font-size:20px;">
<center> <b>2.0 Retrieve
</b></center>     
</div>

In [55]:
from tiny_rag.query.retriever import retrieve

In [63]:
retrieve("高估自我",top_k=1)

[{'id': '/Users/sunshine8641/Code/Tiny_RAG/test_data/人心的本质.pdf#p240#c0',
  'text': '# **提升⾃控⼒的真相** \n\n不仅根本性的⾃我控制是⼀种幻觉，⽽且⽇常的⾃我控制（能动 性的⼀个⽅⾯）也是虚幻的。⼈们想要相信⾃⼰能抗拒诱惑，但⾼⽐ 率的肥胖、酗酒和出轨证明情况相反。⼈们始终⾼估他们坚持完成任 务的⾃律性，低估了完成任务所需的时间（通常低估了⼀半）<sup>29</sup> 。如 果⼈们低估了电击别⼈⽽导致其死亡的可能性，那么他们也会低估项 ⽬的持续时间和预算，会低估他们将吃掉多少巧克⼒也就不⾜为奇 了。 \n\n当然，仅仅知道我们不擅长⾃我控制并不能提供明确的改进⽅ 法。相对于解决问题，⼼理学更善于指出问题，有时看起来就像把点 ⽯成⾦的过程反过来，令⼀切都黯然失⾊。爱情的魔⼒？上帝的神 秘？⾃由意志的体验？所有都是⼤脑的把戏。不过在⾃我控制的领 域，⼼理学在增加能动性⽅⾯确实提供了⼀些很好的⼩建议。概括起 来就是绝不要依赖⾃控。 \n\n这个答案看起来是⼀种推诿，但它与社会⼼理学的理念是⼀致 的。社会⼼理学认为⼈们通常受环境的⽀配，环境通过⼈们⽆法觉察 的暗⽰发挥影响。 \n\n### **⼼智视窗** \n\n实现⾃我控制的最好⽅法就是绝不要让⾃⼰⾝处不得 不进⾏⾃我控制的情境中。相反，你应该专注于构建环 境，这样你就不可能屈服于诱惑了。例如，要避免在家⾥ 吃会让⼈发胖的零⾷的最好办法是根本不要从⾷品店购买 这种零⾷。⽆论半夜你多么渴望那些零⾷，但你⽆法如 愿。你⾄少不得不费劲地开车找24⼩时营业的便利店，在 此期间你的⾷欲会慢慢消退。 \n\n这是StayFocused这类计算机程序背后的原理，它能屏蔽Facebook 等⽹站，这样你就可以专⼼写作了。这些程序的⼯作⽅式主要有两 种。第⼀种是计算你开⼩差的时间并让你注意到这些时间，因为当你 注意到被浪费的时间加起来有好⼏个⼩时时，就不太可能把时间浪费',
  'metadata': {'chunk_index': 0,
   'filename': '人心的本质.pdf',
   'page': 240,
   'source': '/Users/sunshine8641/Code/Tiny_RAG/test_data/


<div class="alert alert-info", style=" padding: 20px;font-size:15px;">
<center> <b> pipeline
</b></center> 

</div>


In [64]:
from tiny_rag.query.pipeline import answer_query

In [66]:
answer_query("人心的本质是什么？", top_k=5,where=None, backend="mock",model=None,max_context_chars=8000)

{'query': '人心的本质是什么？',
 'answer': '[MOCK RESPONSE — no LLM was actually called]\n\n（这里是 LLM 应该回答的内容。当前是 mock 模式。\n要接真实 LLM，设环境变量 OPENAI_API_KEY 并用 backend=\'openai\'，或装 Ollama 跑本地模型用 backend=\'ollama\'。）\n\nPrompt 收到了：\n────────────────────────────────────────\n你是一个知识助手。请严格基于下面的【参考资料】回答用户的问题。\n\n规则：\n1. 只使用【参考资料】中的信息回答，不要编造\n2. 如果【参考资料】里没有答案，直接说"根据现有资料无法回答"\n3. 回答时在末尾用 [1] [2] 等标注引用了哪条资料\n4. 用中文回答（除非用户用其他语言提问）\n\n【参考资料】\n[1] 人心的本质.pdf (p.13):\n下载“湛庐阅读”App， 搜索“⼈⼼的本质”， ...',
 'sources': [{'index': 1,
   'filename': '人心的本质.pdf',
   'title': '人心的本质（中国社会心理学会2020年度心理学好书，一套理解复杂人性和行为的极简思维工具，彻底改变你看待世界和自己的方式，“谷歌效应”提出者、社会心理学泰斗丹尼尔·韦格纳绝笔之作。）',
   'page': 13,
   'source': '/Users/sunshine8641/Code/Tiny_RAG/test_data/人心的本质.pdf',
   'similarity': 0.7255628108978271,
   'snippet': '下载“湛庐阅读”App， 搜索“⼈⼼的本质”， 获取答案。'},
  {'index': 2,
   'filename': '人心的本质.pdf',
   'title': '人心的本质（中国社会心理学会2020年度心理学好书，一套理解复杂人性和行为的极简思维工具，彻底改变你看待世界和自己的方式，“谷歌效应”提出者、社会心理学泰斗丹尼尔·韦格纳绝笔之作。）',
   'page': 6,
   'source': '/Users/sunshi


<div class="alert alert-danger", style=" padding: 20px;font-size:20px;">
<center> <b>3.0 命令行执行
</b></center>     
</div>

In [68]:
!python -m tiny_rag status


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


objc[55287]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x170aa03a8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x17b7243a8). One of the two will be used. Which one is undefined.
objc[55287]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x170aa03f8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x17b7243f8). One of the two will be used. Which one is undefined.
Total chunks: 601
Data dir:     /Users/sunshine8641/Code/Tiny_RAG/data/chroma
Embedding:    BAAI/bge-m3
LLM:          gpt-4o

Sources (4 files):
  /Users/sunshine8641/Code/Tiny_RAG/test_data/test.pdf
  /Users/sunshine8641/Code/Tiny_RAG/test_data/人心的本质.pdf
  /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf
  /User

In [73]:
!python -m tiny_rag query "你能做什么" --model qwen3:8b


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


objc[90935]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x13d1e43a8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14ff5c3a8). One of the two will be used. Which one is undefined.
objc[90935]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x13d1e43f8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14ff5c3f8). One of the two will be used. Which one is undefined.
Loading embedding model: BAAI/bge-m3
  (first run downloads ~2GB, subsequent runs use HF cache)
  Device: cpu
Delete of nonexisting embedding ID: /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf#p1#c0
Delete of nonexisting embedding ID: /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf#p2#c0
Delete o

In [71]:
!python -m tiny_rag remove   /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


objc[56017]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x160f883a8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x168b943a8). One of the two will be used. Which one is undefined.
objc[56017]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x160f883f8) and /opt/anaconda3/envs/torchpy/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x168b943f8). One of the two will be used. Which one is undefined.
Delete of nonexisting embedding ID: /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf#p1#c0
Delete of nonexisting embedding ID: /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf#p2#c0
Delete of nonexisting embedding ID: /Users/sunshine8641/Code/Tiny_RAG/tests/fixtures/test_book.pdf#p3#c0
✓ Deleted 3 c

`python -m <module>` 跟 `python <script>` 走不同代码路径，靠 **`__main__.py`** 这个特殊文件。



## 两种方式对比

```bash
python tiny_rag/cli/main.py   # 方式 1: 跑脚本
python -m tiny_rag            # 方式 2: 跑包
```

### 方式 1: `python script.py`

```
执行:  python tiny_rag/cli/main.py
       ↓
sys.path[0] = tiny_rag/cli/   ← script 所在目录
       ↓
跑 main.py 顶层代码
       ↓
__name__ = "__main__"
       ↓
执行 main() 函数（如果 if __name__ == "__main__" 里有）
```

**问题**：
- 必须知道完整路径
- cwd 不对 import 可能失败
- 跟包管理割裂

### 方式 2: `python -m module`

```
执行:  python -m tiny_rag
       ↓
sys.path[0] = 当前工作目录
       ↓
Python 找 tiny_rag 包
       ↓
跑 tiny_rag/__main__.py（如果存在）
       ↓
__name__ == "__main__"
       ↓
执行 main() 函数
```

**关键**：`__main__.py` 是 Python 的「包的入口文件」，专门给 `-m` 用。

---

## 加了什么

```python
# tiny_rag/__main__.py
from tiny_rag.cli.main import main

if __name__ == "__main__":
    main()
```

3 行代码，作用：
- `python -m tiny_rag` 时 Python 自动找这个文件
- 调 `tiny_rag.cli.main.main()`
- 跟直接 `python -m tiny_rag.cli.main` 等价

---

## argparse 怎么知道子命令

```python
# tiny_rag/cli/main.py
parser = argparse.ArgumentParser(prog="tiny_rag", ...)
sub = parser.add_subparsers(dest="command", required=True)

p_ingest = sub.add_parser("ingest", help="Ingest files into the vector store")
p_ingest.add_argument("path", help="...")
p_ingest.set_defaults(func=cmd_ingest)

# 其他子命令类似
```

`add_subparsers` 是 argparse 的「多命令」机制：
- `tiny_rag ingest <path>` → 路由到 `cmd_ingest`
- `tiny_rag query <query>` → 路由到 `cmd_query`
- 等等

每个子命令 `set_defaults(func=cmd_xxx)`，最后 `args.func(args)` 一句话分发。

---

## 3 种调用方式完整对比

| 方式 | 命令 | 是否需要 cwd 在项目根 |
|---|---|---|
| 跑脚本 | `python tiny_rag/cli/main.py ingest book.pdf` | 是 |
| 跑包 | `python -m tiny_rag ingest book.pdf` | 是（但 import 自动找包） |
| 装成命令 | `tiny_rag ingest book.pdf` | **否**（任何目录） |

**方式 3 最优雅**——`tiny_rag` 像个真正的系统命令，原因是 pyproject.toml 里配了 entry point。

---

## 怎么升级到方式 3（5 行）

在 `pyproject.toml` 里加：

```toml
[project.scripts]
tiny_rag = "tiny_rag.cli.main:main"
```

`pip install -e .` 后：

```bash
tiny_rag ingest book.pdf      # 直接当命令用
tiny_rag query "..."
tiny_rag status
```

`tiny_rag` 自动出现在 `~/.local/bin/`（或者 conda env 的 bin）。


